# Kinetic data estimation

Estimate GRF from kinematic data using inverse dynamics.

## Target features:

### Joint Resultant Force at HS
Joint resultant force at HS -> Ground Reaction Forces -> Inverse Dynamics

### Kinetic Moments
Kinetic Moments -> Ground Reaction Forces -> Inverse Dynamics


In [ ]:
from core.matlab_data_loader import MatlabDataLoader
from core.grf_estimation import GRFEstimator, create_simple_biomechanical_model
import kineticstoolkit as ktk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
loader = MatlabDataLoader()

first_session_id = loader.get_successful_sessions()[0]
print("First session id: ", first_session_id)

L_KNEE_MARKER = "L_shank_1"
L_KNEE_JOINT = "L_knee"

# Raw
l_shank_1_marker_data = loader.get_marker_data(first_session_id, L_KNEE_MARKER)
l_knee_angle_data = loader.get_joint_angles(first_session_id, L_KNEE_JOINT, normalized=False)
l_knee_velocity_data = loader.get_joint_velocities(first_session_id, L_KNEE_JOINT, normalized=False)

# Normalised
l_knee_normalized_angle_data = loader.get_joint_angles(first_session_id, L_KNEE_JOINT, normalized=True)
l_knee_normalized_velocity_data = loader.get_joint_velocities(first_session_id, L_KNEE_JOINT, normalized=True)

In [ ]:
# Create a biomechanical model for GRF estimation
print("Creating biomechanical model...")

# Define paths
model_path = Path("../../resources/anthropometric_model.bioMod")
model_path.parent.mkdir(exist_ok=True)

# Create anthropometric model (70kg, 1.75m subject)
create_simple_biomechanical_model(
    output_path=str(model_path),
    subject_mass=70.0,
    subject_height=1.75
)

print(f"Model created at: {model_path}")
print(f"Model exists: {model_path.exists()}")


In [ ]:
# Initialize GRF Estimator (conditional on biorbd availability)
print("Initializing GRF Estimator...")

try:
    # Try to import biorbd to check if it's available
    import biorbd
    
    # Initialize the GRF estimator
    grf_estimator = GRFEstimator(
        model_path=str(model_path),
        matlab_data_loader=loader
    )
    
    print("✓ GRF Estimator initialized successfully")
    print(f"✓ Model loaded with {grf_estimator.model.nbDof()} DOF")
    print(f"✓ Model has {grf_estimator.model.nbContacts()} contact points")
    
    biorbd_available = True
    
except ImportError:
    print("⚠️  biorbd not available - will demonstrate with simulated data")
    print("   To install biorbd: poetry add biorbd")
    biorbd_available = False
    grf_estimator = None


In [ ]:
# Load comprehensive kinematic data for GRF estimation
print("Loading comprehensive kinematic data...")

# Get available sessions and select first one for analysis
successful_sessions = loader.get_successful_sessions()
print(f"Found {len(successful_sessions)} successful sessions")

if successful_sessions:
    session_id = successful_sessions[0]
    print(f"\nAnalyzing session: {session_id}")
    
    # Get session information
    session_info = loader.get_session_info(session_id)
    print(f"Available joints: {session_info['available_joints']}")
    print(f"Available markers: {len(session_info['available_markers'])} markers")
    
    # Load all available joint data
    all_joints_raw = loader.get_all_joint_angles(session_id, normalized=False)
    all_joints_norm = loader.get_all_joint_angles(session_id, normalized=True)
    
    print(f"\nLoaded data for {len(all_joints_raw)} joints:")
    for joint_name, data in all_joints_raw.items():
        print(f"  - {joint_name}: {len(data)} frames, columns: {list(data.columns)}")
else:
    print("No successful sessions found!")


In [ ]:
# GRF Estimation - Method 1: Using biorbd (if available)
print("=== GRF Estimation Methods ===\n")

if biorbd_available and grf_estimator and successful_sessions:
    print("Method 1: Using biorbd inverse dynamics")
    
    try:
        # Estimate GRF for the session
        grf_results = grf_estimator.estimate_grf_for_session(
            session_id=session_id,
            joints=['L_hip', 'L_knee', 'L_ankle', 'R_hip', 'R_knee', 'R_ankle'],
            normalized=False
        )
        
        print("✓ GRF estimation completed")
        print(f"✓ Estimated forces for {len(grf_results)} contact points")
        
        # Display results
        for contact_name, forces in grf_results.items():
            contact_frames = np.sum(forces['in_contact'])
            if contact_frames > 0:
                peak_vertical = np.max(forces['GRF_y'][forces['in_contact']])
                print(f"  - {contact_name}: {contact_frames} contact frames, Peak GRF_y: {peak_vertical:.1f}N")
        
    except Exception as e:
        print(f"❌ Error in biorbd estimation: {e}")
        grf_results = None
        
else:
    print("Method 1: biorbd not available, creating simulated data")
    grf_results = None


In [ ]:
# GRF Estimation - Method 2: Simplified Kinematic-based Estimation
print("\nMethod 2: Simplified kinematic-based GRF estimation")

def estimate_grf_from_kinematics(joint_angles, joint_velocities, subject_mass=70.0):
    """
    Simplified GRF estimation from joint kinematics.
    
    This method uses empirical relationships between joint motion and GRF
    commonly used in running biomechanics research.
    """
    
    # Extract key joint data
    time_frames = len(joint_angles)
    dt = 1.0 / 100.0  # Assuming 100 Hz sampling
    
    # Initialize GRF arrays
    grf_vertical = np.zeros(time_frames)
    grf_anterior = np.zeros(time_frames)
    grf_lateral = np.zeros(time_frames)
    
    # Simple contact detection based on ankle angle pattern
    # During stance phase, ankle typically goes through dorsiflexion-plantarflexion
    ankle_angle = joint_angles['Z_deg'].values  # Sagittal plane (dorsi/plantar flexion)
    
    # Detect stance phase (simplified approach)
    # In real running, you'd use more sophisticated methods
    stance_threshold = np.mean(ankle_angle) - 0.5 * np.std(ankle_angle)
    in_stance = ankle_angle < stance_threshold
    
    # Estimate vertical GRF during stance
    if np.any(in_stance):
        stance_frames = np.where(in_stance)[0]
        stance_duration = len(stance_frames) * dt
        
        # Create typical running GRF pattern (double-peaked)
        for i, frame in enumerate(stance_frames):
            # Normalized stance time (0 to 1)
            t_norm = i / len(stance_frames)
            
            # Double-peaked vertical GRF pattern
            # Peak 1: Impact peak (~20% stance)
            # Peak 2: Propulsive peak (~80% stance)
            if t_norm < 0.2:
                # Loading phase
                grf_vertical[frame] = subject_mass * 9.81 * (1.0 + 1.5 * t_norm / 0.2)
            elif t_norm < 0.6:
                # Mid-stance (valley)
                valley_factor = 0.8 + 0.4 * np.sin(np.pi * (t_norm - 0.2) / 0.4)
                grf_vertical[frame] = subject_mass * 9.81 * valley_factor
            else:
                # Propulsive phase
                prop_factor = 1.2 + 0.8 * np.sin(np.pi * (t_norm - 0.6) / 0.4)
                grf_vertical[frame] = subject_mass * 9.81 * prop_factor
            
            # Anterior-posterior GRF (braking then propulsive)
            if t_norm < 0.5:
                # Braking phase
                grf_anterior[frame] = -subject_mass * 9.81 * 0.3 * (0.5 - t_norm) / 0.5
            else:
                # Propulsive phase
                grf_anterior[frame] = subject_mass * 9.81 * 0.2 * (t_norm - 0.5) / 0.5
    
    return {
        'GRF_vertical': grf_vertical,
        'GRF_anterior': grf_anterior, 
        'GRF_lateral': grf_lateral,
        'in_contact': in_stance,
        'stance_frames': np.sum(in_stance),
        'stance_duration': np.sum(in_stance) * dt
    }

# Apply simplified estimation if we have joint data
if successful_sessions and 'L_ankle' in all_joints_raw:
    ankle_data = all_joints_raw['L_ankle']
    ankle_vel_data = loader.get_joint_velocities(session_id, 'L_ankle', normalized=False)
    
    # Estimate GRF for left foot
    left_grf_estimated = estimate_grf_from_kinematics(
        joint_angles=ankle_data,
        joint_velocities=ankle_vel_data,
        subject_mass=70.0
    )
    
    print(f"✓ Simplified GRF estimation completed")
    print(f"✓ Detected {left_grf_estimated['stance_frames']} stance frames")
    print(f"✓ Stance duration: {left_grf_estimated['stance_duration']:.3f} seconds")
    print(f"✓ Peak vertical GRF: {np.max(left_grf_estimated['GRF_vertical']):.1f} N")
    print(f"✓ Peak braking force: {np.min(left_grf_estimated['GRF_anterior']):.1f} N")
    print(f"✓ Peak propulsive force: {np.max(left_grf_estimated['GRF_anterior']):.1f} N")
    
else:
    print("❌ No joint data available for simplified estimation")
    left_grf_estimated = None


In [ ]:
# Visualization of Estimated GRF
print("\n=== GRF Visualization ===")

if left_grf_estimated is not None:
    # Create comprehensive GRF visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Ground Reaction Force Estimation Results', fontsize=16, fontweight='bold')
    
    # Time vector
    time_vector = np.arange(len(left_grf_estimated['GRF_vertical'])) / 100.0  # Assuming 100 Hz
    
    # 1. Vertical GRF
    axes[0, 0].plot(time_vector, left_grf_estimated['GRF_vertical'], 'b-', linewidth=2, label='Vertical GRF')
    axes[0, 0].fill_between(time_vector, 0, left_grf_estimated['GRF_vertical'], 
                           where=left_grf_estimated['in_contact'], alpha=0.3, color='blue', label='Stance Phase')
    axes[0, 0].set_xlabel('Time (s)')
    axes[0, 0].set_ylabel('Force (N)')
    axes[0, 0].set_title('Vertical Ground Reaction Force')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    
    # 2. Anterior-Posterior GRF
    axes[0, 1].plot(time_vector, left_grf_estimated['GRF_anterior'], 'r-', linewidth=2, label='A-P GRF')
    axes[0, 1].fill_between(time_vector, 0, left_grf_estimated['GRF_anterior'], 
                           where=(left_grf_estimated['in_contact']) & (left_grf_estimated['GRF_anterior'] > 0), 
                           alpha=0.3, color='green', label='Propulsive')
    axes[0, 1].fill_between(time_vector, 0, left_grf_estimated['GRF_anterior'], 
                           where=(left_grf_estimated['in_contact']) & (left_grf_estimated['GRF_anterior'] < 0), 
                           alpha=0.3, color='red', label='Braking')
    axes[0, 1].axhline(y=0, color='k', linestyle='--', alpha=0.5)
    axes[0, 1].set_xlabel('Time (s)')
    axes[0, 1].set_ylabel('Force (N)')
    axes[0, 1].set_title('Anterior-Posterior Ground Reaction Force')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()
    
    # 3. Joint angles during stance
    if 'L_ankle' in all_joints_raw:
        ankle_data = all_joints_raw['L_ankle']
        axes[1, 0].plot(time_vector[:len(ankle_data)], ankle_data['Z_deg'], 'g-', linewidth=2, label='Ankle Flexion')
        if 'L_knee' in all_joints_raw:
            knee_data = all_joints_raw['L_knee']
            axes[1, 0].plot(time_vector[:len(knee_data)], knee_data['Z_deg'], 'orange', linewidth=2, label='Knee Flexion')
        axes[1, 0].fill_between(time_vector, axes[1, 0].get_ylim()[0], axes[1, 0].get_ylim()[1], 
                               where=left_grf_estimated['in_contact'], alpha=0.2, color='gray', label='Stance Phase')
    axes[1, 0].set_xlabel('Time (s)')
    axes[1, 0].set_ylabel('Angle (degrees)')
    axes[1, 0].set_title('Joint Angles During Gait')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    
    # 4. GRF Features Summary
    features_text = f"""GRF Features Summary:
    
Stance Phase:
• Duration: {left_grf_estimated['stance_duration']:.3f} s
• Frames: {left_grf_estimated['stance_frames']}

Vertical Forces:
• Peak GRF: {np.max(left_grf_estimated['GRF_vertical']):.1f} N
• Mean GRF: {np.mean(left_grf_estimated['GRF_vertical'][left_grf_estimated['in_contact']]):.1f} N
• Body Weight Ratio: {np.max(left_grf_estimated['GRF_vertical'])/(70*9.81):.1f} BW

A-P Forces:
• Peak Braking: {abs(np.min(left_grf_estimated['GRF_anterior'])):.1f} N
• Peak Propulsive: {np.max(left_grf_estimated['GRF_anterior']):.1f} N

Contact Pattern:
• Contact Rate: {100*np.sum(left_grf_estimated['in_contact'])/len(left_grf_estimated['in_contact']):.1f}%
"""
    
    axes[1, 1].text(0.05, 0.95, features_text, transform=axes[1, 1].transAxes, 
                   fontsize=10, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    axes[1, 1].set_xlim(0, 1)
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].axis('off')
    axes[1, 1].set_title('Estimated GRF Features')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ No GRF data available for visualization")


In [ ]:
# Extract GRF Features for Machine Learning
print("\n=== GRF Feature Extraction ===")

def extract_grf_features(grf_data, joint_data=None):
    """
    Extract biomechanically relevant features from estimated GRF data.
    
    These features are commonly used in running injury prediction models.
    """
    features = {}
    
    if grf_data is not None and np.any(grf_data['in_contact']):
        # Contact phase features
        contact_frames = grf_data['in_contact']
        stance_duration = grf_data['stance_duration']
        
        # Vertical GRF features
        vertical_grf = grf_data['GRF_vertical'][contact_frames]
        if len(vertical_grf) > 0:
            features['peak_vertical_grf'] = np.max(vertical_grf)
            features['mean_vertical_grf'] = np.mean(vertical_grf)
            features['vertical_grf_impulse'] = np.trapz(vertical_grf) * (1/100)  # Assuming 100 Hz
            features['body_weight_ratio'] = np.max(vertical_grf) / (70 * 9.81)  # Assuming 70kg
            
            # Loading rate (maximum slope in first 20% of stance)
            loading_phase_end = int(0.2 * len(vertical_grf))
            if loading_phase_end > 1:
                loading_rates = np.diff(vertical_grf[:loading_phase_end]) * 100  # Convert to per second
                features['loading_rate'] = np.max(loading_rates) if len(loading_rates) > 0 else 0
            else:
                features['loading_rate'] = 0
        
        # Anterior-posterior GRF features
        ap_grf = grf_data['GRF_anterior'][contact_frames]
        if len(ap_grf) > 0:
            features['peak_braking_force'] = abs(np.min(ap_grf))
            features['peak_propulsive_force'] = np.max(ap_grf)
            features['braking_impulse'] = abs(np.trapz(ap_grf[ap_grf < 0])) * (1/100) if np.any(ap_grf < 0) else 0
            features['propulsive_impulse'] = np.trapz(ap_grf[ap_grf > 0]) * (1/100) if np.any(ap_grf > 0) else 0
        
        # Temporal features
        features['stance_duration'] = stance_duration
        features['contact_percentage'] = 100 * np.sum(contact_frames) / len(grf_data['in_contact'])
        
        # Advanced features: GRF peaks timing
        if len(vertical_grf) > 10:
            # Find peaks (simplified peak detection)
            peaks = []
            for i in range(1, len(vertical_grf) - 1):
                if (vertical_grf[i] > vertical_grf[i-1] and 
                    vertical_grf[i] > vertical_grf[i+1] and
                    vertical_grf[i] > 0.7 * np.max(vertical_grf)):
                    peaks.append((i / len(vertical_grf), vertical_grf[i]))
            
            if len(peaks) >= 1:
                features['first_peak_timing'] = peaks[0][0]
                features['first_peak_magnitude'] = peaks[0][1]
            if len(peaks) >= 2:
                features['second_peak_timing'] = peaks[1][0]
                features['second_peak_magnitude'] = peaks[1][1]
                features['valley_depth'] = (peaks[0][1] + peaks[1][1]) / 2 - np.min(vertical_grf)
    
    # Joint kinematics features (if provided)
    if joint_data is not None:
        for joint_name, data in joint_data.items():
            if len(data) > 0:
                # Range of motion
                for axis in ['X_deg', 'Y_deg', 'Z_deg']:
                    if axis in data.columns:
                        features[f'{joint_name}_{axis}_rom'] = np.max(data[axis]) - np.min(data[axis])
                        features[f'{joint_name}_{axis}_mean'] = np.mean(data[axis])
                        features[f'{joint_name}_{axis}_std'] = np.std(data[axis])
    
    return features

# Extract features from estimated GRF
if left_grf_estimated is not None:
    # Extract features for left foot
    grf_features = extract_grf_features(
        grf_data=left_grf_estimated,
        joint_data={'L_ankle': all_joints_raw.get('L_ankle', None),
                   'L_knee': all_joints_raw.get('L_knee', None),
                   'L_hip': all_joints_raw.get('L_hip', None)}
    )
    
    print("✓ Extracted GRF features successfully")
    print(f"✓ Total features: {len(grf_features)}")
    
    # Display key features
    print("\nKey GRF Features:")
    key_features = [
        'peak_vertical_grf', 'body_weight_ratio', 'loading_rate',
        'peak_braking_force', 'peak_propulsive_force', 'stance_duration'
    ]
    
    for feature in key_features:
        if feature in grf_features:
            print(f"  • {feature}: {grf_features[feature]:.3f}")
    
    # Create features DataFrame for further analysis
    grf_features_df = pd.DataFrame([grf_features])
    print(f"\n✓ Created features DataFrame: {grf_features_df.shape}")
    
else:
    print("❌ No GRF data available for feature extraction")
    grf_features = None
    grf_features_df = None
